# 逻辑回归第四课：手写逻辑回归

这一课的目标是把前面三课真正连起来，用 NumPy 从零实现一个最小版逻辑回归。

这份 Notebook 重点包括：

- 从数据到概率预测
- 交叉熵损失的计算
- 梯度的向量化实现
- 梯度下降训练参数
- 从概率到类别预测


## 1. 先回顾完整主线

逻辑回归的训练主线可以压缩成下面几步：

$$
z = Xw + b
$$

$$
p = \sigma(z) = \frac{1}{1+e^{-z}}
$$

$$
J(w,b) = -\frac{1}{m}\sum_{i=1}^{m} [ y^{(i)}\log p^{(i)} + (1-y^{(i)})\log(1-p^{(i)}) ]
$$

$$
\nabla_w J = \frac{1}{m}X^T(p-y)
$$

$$
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m}(p^{(i)}-y^{(i)})
$$

$$
w := w - \alpha \nabla_w J
$$

$$
b := b - \alpha \frac{\partial J}{\partial b}
$$

这一课就是把这些公式全部落到代码里。

## 2. 先准备一个最小数据集

为了先把训练流程看清楚，这里不用真实大数据集，而是先构造一个非常小的二分类数据集。

我们让样本有两个特征：

- 第一个特征是会变化的数值
- 第二个特征恒为 1，可以把它理解成一个辅助特征

标签只有 0 和 1。

后面你会看到：即使是这样的小数据集，逻辑回归的完整训练流程已经完全具备了。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X = np.array([
    [0.0, 1.0],
    [0.5, 1.0],
    [1.0, 1.0],
    [1.5, 1.0],
    [2.0, 1.0],
    [2.5, 1.0],
    [3.0, 1.0],
    [3.5, 1.0]
])

y = np.array([0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0])

print('X shape =', X.shape)
print('y shape =', y.shape)
print(X)
print(y)


## 3. 先写最基本的函数

我们需要 5 个基础函数：

- `sigmoid`：把线性分数变成概率
- `compute_loss`：计算平均交叉熵
- `compute_gradients`：计算 $dw$ 和 $db$
- `predict_proba`：输出正类概率
- `predict`：把概率变成类别

这里的实现全部使用向量化写法，不用逐样本循环。

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def compute_loss(y, p):
    eps = 1e-12
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def compute_gradients(X, y, w, b):
    m = len(y)
    z = X @ w + b
    p = sigmoid(z)

    dw = (X.T @ (p - y)) / m
    db = np.mean(p - y)
    return dw, db

def predict_proba(X, w, b):
    z = X @ w + b
    return sigmoid(z)

def predict(X, w, b, threshold=0.5):
    p = predict_proba(X, w, b)
    return (p >= threshold).astype(int)


## 4. 检查梯度函数在做什么

在训练开始前，先用初始参数看一眼梯度。

一开始把参数设成：

$$
w = 0, \qquad b = 0
$$

这时所有样本的预测概率都会是 0.5。

如果梯度函数实现正确，那么我们应该能算出一个明确的更新方向。

In [ ]:
w = np.zeros(X.shape[1])
b = 0.0

p0 = predict_proba(X, w, b)
loss0 = compute_loss(y, p0)
dw0, db0 = compute_gradients(X, y, w, b)

print('initial probabilities =', p0)
print('initial loss =', loss0)
print('initial dw =', dw0)
print('initial db =', db0)


## 5. 写训练函数

现在把前面的部件拼起来，写一个最小版训练函数。

每一轮训练都做 4 件事：

1. 前向计算 $p$
2. 计算 loss
3. 计算梯度 $dw, db$
4. 更新参数 $w, b$

同时把每一轮的损失保存下来，后面画曲线。

In [ ]:
def fit_logistic_regression(X, y, lr=0.1, epochs=500):
    n_features = X.shape[1]
    w = np.zeros(n_features)
    b = 0.0
    loss_history = []

    for epoch in range(epochs):
        p = predict_proba(X, w, b)
        loss = compute_loss(y, p)
        dw, db = compute_gradients(X, y, w, b)

        w -= lr * dw
        b -= lr * db

        loss_history.append(loss)

        if epoch % 50 == 0 or epoch == epochs - 1:
            print(f'epoch={epoch:3d}, loss={loss:.6f}, w={w}, b={b:.6f}')

    return w, b, loss_history


## 6. 开始训练

现在真正训练模型。

如果实现正确，你应该会看到：

- loss 持续下降
- 参数不断更新
- 最终预测结果和标签基本一致

In [ ]:
w, b, loss_history = fit_logistic_regression(X, y, lr=0.3, epochs=500)

print('\nfinal w =', w)
print('final b =', b)


## 7. 看训练后的概率和类别预测

训练完成后，我们分别看：

- 每个样本的正类概率
- 最终的类别预测
- 是否和真实标签一致

In [ ]:
final_p = predict_proba(X, w, b)
final_pred = predict(X, w, b)

for i in range(len(y)):
    print(
        f'sample={i}, x={X[i]}, y={int(y[i])}, '
        f'p={final_p[i]:.6f}, pred={final_pred[i]}'
    )


## 8. 画出 loss 曲线

训练一个模型时，最先应该看的不是准确率，而是 loss 是否稳定下降。

如果 loss 一直下降，通常说明：

- 梯度方向基本正确
- 学习率没有明显失控
- 训练过程是有效的

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Logistic Regression Training Loss')
plt.grid(True)
plt.show()


## 9. 这份手写代码到底对应了哪些公式

把代码和公式一一对应起来：

前向计算：

$$
z = Xw + b
$$

$$
p = \sigma(z)
$$

损失函数：

$$
J(w,b) = -\frac{1}{m}\sum_{i=1}^{m} [ y^{(i)}\log p^{(i)} + (1-y^{(i)})\log(1-p^{(i)}) ]
$$

梯度：

$$
\nabla_w J = \frac{1}{m}X^T(p-y)
$$

$$
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m}(p^{(i)}-y^{(i)})
$$

参数更新：

$$
w := w - \alpha \nabla_w J
$$

$$
b := b - \alpha \frac{\partial J}{\partial b}
$$

到这里为止，你已经不只是“会用逻辑回归”，而是已经知道它训练时内部到底在算什么。

## 10. 本节小结

这一课完成了三件关键事情：

- 把逻辑回归的数学公式变成了代码
- 用梯度下降真正训练了参数
- 验证了 loss 会下降、概率会逐渐合理

下一步最自然的内容就是：

- 用 `sklearn` 的 `LogisticRegression` 做标准工作流
- 学习 `predict_proba`、`predict`、`decision_function`
- 看真实数据集上的效果
